# 🎲 VibeMV 3D Model Generator\n\nGenerate 3D models from your VibeMV timeline using **TripoSR**.\n\n## 📋 Setup Checklist\n\n- [ ] Enable GPU: Runtime → Change runtime type → T4 GPU\n- [ ] Have your VibeMV timeline JSON ready\n- [ ] Run all cells in order\n\n**Estimated time:** 5-10 minutes for 3-5 scenes

In [ ]:
# @title ✅ Check GPU Availability\nimport torch\n\nif torch.cuda.is_available():\n    print(f\"✅ GPU Detected: {torch.cuda.get_device_name(0)}\")\n    print(f\"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\")\nelse:\n    print(\"❌ No GPU detected!\")\n    print(\"⚠️  Please enable GPU: Runtime → Change runtime type → T4 GPU\")

In [ ]:
# @title 📦 Install Dependencies (2-3 minutes)\n%%capture\n\nprint(\"Installing TripoSR and dependencies...\")\n\n# Install core packages\n!pip install -q torch torchvision\n!pip install -q diffusers transformers accelerate\n!pip install -q trimesh\n!pip install -q rembg\n!pip install -q imageio pillow\n\n# Install TripoSR\n!pip install -q git+https://github.com/VAST-AI-Research/TripoSR.git\n\nprint(\"✅ All dependencies installed!\")

In [ ]:
# @title 📤 Upload VibeMV Timeline\nfrom google.colab import files\nimport json\n\nprint(\"📁 Please upload your vibemv_timeline.json...\")\nuploaded = files.upload()\n\ntimeline_file = list(uploaded.keys())[0]\nwith open(timeline_file, 'r') as f:\n    timeline = json.load(f)\n\nprint(f\"\\n✅ Loaded {len(timeline['scenes'])} scenes:\")\nfor i, scene in enumerate(timeline['scenes']):\n    print(f\"  {i+1}. {scene['prompt'][:60]}...\")

In [ ]:
# @title 🎨 Generate Images from Prompts (10-15s per scene)\nimport torch\nfrom diffusers import StableDiffusionXLPipeline\nimport os\n\nos.makedirs('generated_images', exist_ok=True)\n\nprint(\"Loading Stable Diffusion XL...\")\npipe = StableDiffusionXLPipeline.from_pretrained(\n    \"stabilityai/stable-diffusion-xl-base-1.0\",\n    torch_dtype=torch.float16,\n    variant=\"fp16\"\n).to(\"cuda\")\n\nscene_images = []\nprint(\"\\n🎨 Generating images...\\n\")\n\nfor i, scene in enumerate(timeline['scenes']):\n    print(f\"Scene {i+1}/{len(timeline['scenes'])}: {scene['prompt'][:50]}...\")\n    \n    image = pipe(\n        prompt=scene['prompt'],\n        num_inference_steps=30,\n        height=512,\n        width=512\n    ).images[0]\n    \n    img_path = f\"generated_images/scene_{i:03d}.png\"\n    image.save(img_path)\n    scene_images.append(img_path)\n    print(f\"  ✅ Saved {img_path}\")\n\ndel pipe\ntorch.cuda.empty_cache()\nprint(f\"\\n✅ Generated {len(scene_images)} images!\")

In [ ]:
# @title 🎲 Generate 3D Models with TripoSR (30-60s per model)\nfrom tsr.system import TSR\nfrom tsr.utils import remove_background, resize_foreground\nfrom PIL import Image\nimport rembg\nimport trimesh\n\nos.makedirs('3d_models', exist_ok=True)\n\nprint(\"Loading TripoSR...\")\nmodel = TSR.from_pretrained(\n    \"stabilityai/TripoSR\",\n    config_name=\"config.yaml\",\n    weight_name=\"model.ckpt\"\n)\nmodel.to(\"cuda\")\n\n# Background remover\nrembg_session = rembg.new_session()\n\nprint(\"\\n🎲 Converting images to 3D models...\\n\")\n\nmodel_paths = []\n\nfor i, img_path in enumerate(scene_images):\n    print(f\"Processing model {i+1}/{len(scene_images)}...\")\n    \n    # Load and preprocess image\n    image = Image.open(img_path)\n    image = remove_background(image, rembg_session)\n    image = resize_foreground(image, 0.85)\n    \n    # Generate 3D model\n    with torch.no_grad():\n        scene_codes = model([image], device=\"cuda\")\n    \n    # Extract mesh\n    meshes = model.extract_mesh(scene_codes)\n    mesh = meshes[0]\n    \n    # Save as OBJ\n    obj_path = f\"3d_models/scene_{i:03d}.obj\"\n    mesh.export(obj_path)\n    \n    # Save as GLB\n    glb_path = f\"3d_models/scene_{i:03d}.glb\"\n    mesh.export(glb_path)\n    \n    model_paths.append((obj_path, glb_path))\n    \n    print(f\"  ✅ Generated {obj_path} and {glb_path}\")\n\nprint(f\"\\n✅ Generated {len(model_paths)} 3D models!\")

In [ ]:
# @title 👁️ Preview 3D Models\nimport plotly.graph_objects as go\n\nprint(\"Loading first model for preview...\")\n\nmesh = trimesh.load(model_paths[0][0])\nvertices = mesh.vertices\nfaces = mesh.faces\n\nfig = go.Figure(data=[\n    go.Mesh3d(\n        x=vertices[:, 0],\n        y=vertices[:, 1],\n        z=vertices[:, 2],\n        i=faces[:, 0],\n        j=faces[:, 1],\n        k=faces[:, 2],\n        opacity=0.8\n    )\n])\n\nfig.update_layout(\n    scene=dict(\n        aspectmode='data',\n        camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))\n    ),\n    title=\"3D Model Preview (Scene 1)\"\n)\n\nfig.show()

In [ ]:
# @title 📥 Download All 3D Models\nimport shutil\n\n# Create ZIP archive\narchive_name = 'vibemv_3d_models'\nshutil.make_archive(archive_name, 'zip', '3d_models')\n\nprint(f\"📦 Created {archive_name}.zip with {len(model_paths)} models\")\nprint(\"\\nDownloading...\")\n\nfiles.download(f'{archive_name}.zip')\n\nprint(\"\\n✅ Download complete!\")\nprint(\"\\n📝 What's included:\")\nfor i, (obj, glb) in enumerate(model_paths):\n    print(f\"  Scene {i+1}: {obj} + {glb}\")

---\n\n## 🎉 Next Steps\n\n1. **Import to Blender**: Use the .obj files\n2. **Web viewer**: Upload .glb to https://gltf-viewer.donmccurdy.com/\n3. **Animate**: Add keyframes and camera paths in Blender\n4. **Render**: Export frames and use in your video\n\n## 💡 Tips\n\n- Lower VRAM? Reduce image size to 256x256\n- Need better quality? Try Colab Pro for A100 GPU\n- Want animation? Export 3D → Blender → Animate → Render